In [16]:
#Develop a model for the multi linear regression using the given data set.
#DailyDelhiClimateTest.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv('DailyDelhiClimateTest.csv')

#printing the header
print(df.head())

#date to numeric 
df['date']=pd.to_datetime(df['date'])
df['date']=df['date'].map(pd.Timestamp.toordinal)

#prepare data
X=df[['date','humidity','wind_speed','meanpressure']]
y=df['meantemp']

#split the training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#printing it 
print("Training size:",X_train.shape)
print("Testing size:",X_test.shape)

#missing values 
print(df.isnull().sum())

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluation
print("R2 Score:", round(r2_score(y_test, y_pred),2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)),2))

#printing coeff and intercept
print("Intercept:", model.intercept_)
print("Coefficients:", dict(zip(['humidity', 'wind_speed', 'meanpressure'], model.coef_)))



         date   meantemp   humidity  wind_speed  meanpressure
0  2017-01-01  15.913043  85.869565    2.743478     59.000000
1  2017-01-02  18.500000  77.222222    2.894444   1018.277778
2  2017-01-03  17.111111  81.888889    4.016667   1018.333333
3  2017-01-04  18.700000  70.050000    4.545000   1015.700000
4  2017-01-05  18.388889  74.944444    3.300000   1014.333333
Training size: (91, 4)
Testing size: (23, 4)
date            0
meantemp        0
humidity        0
wind_speed      0
meanpressure    0
dtype: int64
R2 Score: -190.49
RMSE: 86.35
Intercept: 22.058675572662096
Coefficients: {'humidity': 2.227191384948631, 'wind_speed': -1.5415007377826848, 'meanpressure': -0.19695950382497485}


In [6]:
#hand coded 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load and inspect
df = pd.read_csv('DailyDelhiClimateTest.csv')
print("Dataset Preview:")
print(df.head())
print("\nNull Values:\n", df.isnull().sum())

# Preprocessing
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['meantemp'] = pd.to_numeric(df['meantemp'], errors='coerce')
df['humidity'] = pd.to_numeric(df['humidity'], errors='coerce')
df['wind_speed'] = pd.to_numeric(df['wind_speed'], errors='coerce')
df['meanpressure'] = pd.to_numeric(df['meanpressure'], errors='coerce')
df.dropna(inplace=True)

# Days elapsed as numeric date feature
df['days_elapsed'] = (df['date'] - df['date'].min()).dt.days

# Features and target
X = df[['days_elapsed', 'humidity', 'wind_speed', 'meanpressure']].values
y = df['meantemp'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class MultipleLinearRegression:

    def __init__(self):
        self.weights = None

    def _add_bias(self, X):
        bias = np.ones((X.shape[0], 1))
        return np.column_stack((bias, X))

    def fit(self, X, y):
        X_b = self._add_bias(X)
        y = y.reshape(-1, 1)
        self.weights = np.linalg.inv(X_b.T @ X_b) @ (X_b.T @ y)

    def predict(self, X):
        X_b = self._add_bias(X)
        return (X_b @ self.weights).flatten()

    def r2_score(self, y_actual, y_predicted):
        ss_res = np.sum((y_actual - y_predicted) ** 2)
        ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
        return round(1 - (ss_res / ss_tot), 2)


model = MultipleLinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = model.r2_score(y_test, y_pred)
print("R2 Score:", r2)

Dataset Preview:
         date   meantemp   humidity  wind_speed  meanpressure
0  2017-01-01  15.913043  85.869565    2.743478     59.000000
1  2017-01-02  18.500000  77.222222    2.894444   1018.277778
2  2017-01-03  17.111111  81.888889    4.016667   1018.333333
3  2017-01-04  18.700000  70.050000    4.545000   1015.700000
4  2017-01-05  18.388889  74.944444    3.300000   1014.333333

Null Values:
 date            0
meantemp        0
humidity        0
wind_speed      0
meanpressure    0
dtype: int64
R2 Score: -190.49
